
# Free-Free Absorption of a Synchrotron SED

In radio supernovae and GRB afterglows, the synchrotron-emitting shock is
often surrounded by ionized circumstellar material (CSM) shed by the
progenitor star. This CSM produces **free-free absorption (FFA)**, which
attenuates the low-frequency synchrotron emission:

\begin{align}F_\nu^{\rm obs} = F_\nu^{\rm synch} \exp\!\left[-\tau_{\rm ff}(\nu)\right].\end{align}

For a steady stellar wind the free-free optical depth scales as
$\tau_{\rm ff} \propto \dot{M}^2 v_w^{-2} \nu^{-2}$, so denser
or slower winds push the FFA turnover to higher frequencies and can
completely obscure the SSA peak during the first weeks to months after
explosion.

We build the intrinsic SSA synchrotron SED with
:class:`~triceratops.radiation.synchrotron.SEDs.one_zone.PowerLaw_SSA_SynchrotronSED`,
compute the wind optical depth with
:func:`~triceratops.radiation.free_free.absorption.compute_ff_RJ_optical_depth_wind_Mdot`,
and evolve both quantities over a simple power-law expansion to show how
the double-turnover structure changes with time.

## Relevant API
- :class:`~triceratops.radiation.synchrotron.SEDs.one_zone.PowerLaw_SSA_SynchrotronSED`
- :func:`~triceratops.radiation.free_free.absorption.compute_ff_RJ_optical_depth_wind_Mdot`


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from matplotlib.colors import LogNorm

from triceratops.radiation.free_free import compute_ff_RJ_optical_depth_wind_Mdot
from triceratops.radiation.synchrotron import PowerLaw_SSA_SynchrotronSED
from triceratops.utils.plot_utils import set_plot_style

set_plot_style()

## Intrinsic Synchrotron Spectrum

:class:`~triceratops.radiation.synchrotron.SEDs.one_zone.PowerLaw_SSA_SynchrotronSED`
maps physical parameters ($B$, $R$, microphysics fractions)
to the phenomenological SED parameters required by
:meth:`~triceratops.radiation.synchrotron.SEDs.one_zone.PowerLaw_SSA_SynchrotronSED.sed`
via a one-zone closure relation.



In [ ]:
# Create the frequency grid on which we'll evaluate the SED.
nu = np.geomspace(1e7, 1e12, 1000) * u.Hz

# Physical source parameters
B = 1.0 * u.G  # magnetic field strength
R = 5e16 * u.cm  # source radius
D_L = 50.0 * u.Mpc  # luminosity distance (typical nearby SN)
p = 3.0  # electron power-law index

# Compute the SED.
sed = PowerLaw_SSA_SynchrotronSED()
params = sed.from_physics_to_params(
    B=B,
    R=R,
    gamma_min=1.0,
    p=p,
    epsilon_E=0.1,
    epsilon_B=0.1,
    luminosity_distance=D_L,
    pitch_average=True,
)

F_synch = sed.sed(
    nu=nu,
    nu_m=params["nu_m"],
    F_norm=params["F_norm"],
    omega=params["omega"],
    p=p,
)

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.loglog(nu.to_value(u.GHz), F_synch.to_value("mJy"), lw=2, label="Synchrotron SED (SSA)", color="k")
ax.axvline(
    params["nu_a"].to_value(u.GHz),
    ls=":",
    color="k",
    alpha=0.8,
    label=rf"$\nu_a = {params['nu_a'].to_value(u.GHz):.2f}$ GHz",
)

ax.set(
    xlabel="Frequency (GHz)",
    ylabel=r"$F_\nu$ (mJy)",
)
ax.legend(fontsize=9)
ax.grid(True, which="both", ls="--", alpha=0.3)
ax.set_ylim([1e-3, 1e3])
ax.set_xlim([1e-1, 1e3])
plt.tight_layout()
plt.show()

## Free-Free Optical Depth

For a steady stellar wind the density profile is

\begin{align}\rho(r) = \frac{\dot{M}}{4\pi\,r^2\,v_w},\end{align}

which gives $\tau_{\rm ff} \propto \nu^{-2}$.
:func:`~triceratops.radiation.free_free.absorption.compute_ff_RJ_optical_depth_wind_Mdot`
accepts $\dot{M}$ and $v_w$ directly and converts to
number densities at the inner radius internally.



In [ ]:
# Set up the parameters
MASS_LOSS_RATE = 1e-6 * u.Msun / u.yr  # mass loss rate
WIND_SPEED = 1000 * u.km / u.s  # wind speed
T_WIND = 1e4 * u.K  # wind temperature

# Compute the optical depths.
tau_ff = compute_ff_RJ_optical_depth_wind_Mdot(
    nu,
    Mdot=MASS_LOSS_RATE,
    v_w=WIND_SPEED,
    r_min=R,
    mu_e=1.0,
    mu_i=1.0,
    temperature=T_WIND,
    Z=1,
    thomson=False,
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.loglog(nu.to_value(u.GHz), tau_ff, ls="-", color="k", label="Free-Free Optical Depth")
ax.set(
    xlabel="Frequency (GHz)",
    ylabel=r"$\tau_{\rm ff}$",
    title="Free-Free Optical Depth (Wind CSM)",
)
plt.tight_layout()
plt.show()

## Apply Free-Free Absorption

We now compute the observed spectrum:

\begin{align}F_\nu^{\rm obs} = F_\nu^{\rm synch} e^{-\tau_{\rm ff}}\end{align}



In [ ]:
# Apply the attenuation.
F_obs = F_synch * np.exp(-tau_ff)


# Plot the result.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.loglog(nu.to_value(u.GHz), F_synch.to_value("mJy"), lw=2, label="Synchrotron SSA", color="k")
ax.loglog(nu.to_value(u.GHz), F_obs.to_value("mJy"), lw=2, label="Synchrotron SSA+FFA", color="darkred")

ax.set(
    xlabel="Frequency (GHz)",
    ylabel=r"$F_\nu$ (mJy)",
    title="Effect of Free-Free Absorption",
    ylim=(1e-1, 5e2),
    xlim=(1e0, 1e3),
)

ax.legend()
ax.grid(True, which="both", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()

## Time Dependence

We now evolve the system over time by treating both $B$ and $R$
as power laws in time — a common approximation for freely-expanding shocks.
As the shock radius grows the wind optical depth decreases, causing the FFA
turnover to sweep downward in frequency and eventually reveal the SSA peak.



In [ ]:
nu = np.geomspace(1e7, 1e13, 2000) * u.Hz

TIMES = np.geomspace(10, 1_000, 5) * u.day

BETA, GAMMA = -0.8, 0.75  # power-law indices: B ∝ t^β, R ∝ t^γ
B_0 = 1.0 * u.G
R_0 = 5e16 * u.cm

# Normalised to t = 100 days.
B_t = B_0 * (TIMES / (100 * u.day)) ** BETA
R_t = R_0 * (TIMES / (100 * u.day)) ** GAMMA

params = [
    sed.from_physics_to_params(
        B=B,
        R=R,
        gamma_min=1.0,
        p=p,
        epsilon_E=0.1,
        epsilon_B=0.1,
        luminosity_distance=D_L,
        pitch_average=True,
    )
    for (B, R) in zip(B_t, R_t)
]

F_synch_t = np.zeros((len(TIMES), len(nu)))
for i, param in enumerate(params):
    F_synch_t[i] = sed.sed(
        nu=nu,
        nu_m=param["nu_m"],
        F_norm=param["F_norm"],
        omega=param["omega"],
        p=p,
    ).to_value("mJy")

tau_ffs = [
    compute_ff_RJ_optical_depth_wind_Mdot(
        nu,
        Mdot=MASS_LOSS_RATE,
        v_w=WIND_SPEED,
        r_min=R,
        mu_e=1.0,
        mu_i=1.0,
        temperature=T_WIND,
        Z=1,
        thomson=False,
    )
    for R in R_t
]

F_obs_t = np.zeros_like(F_synch_t)
for i in range(len(TIMES)):
    F_obs_t[i, :] = F_synch_t[i, :] * np.exp(-tau_ffs[i])

F_synch_t = F_synch_t * u.mJy
F_obs_t = F_obs_t * u.mJy

fig, ax = plt.subplots(figsize=(8, 5))
cmap = plt.get_cmap("plasma")
norm = LogNorm(TIMES.min().value, TIMES.max().value)

for i, time in enumerate(TIMES):
    color = cmap(norm(time.value))
    ax.loglog(
        nu.to_value(u.GHz),
        F_synch_t[i].to_value("mJy"),
        lw=1.5,
        ls="--",
        color=color,
        label=f"Intrinsic SSA  (t = {time.value:.0f} d)",
    )
    ax.loglog(
        nu.to_value(u.GHz), F_obs_t[i].to_value("mJy"), lw=2, color=color, label=f"SSA + FFA  (t = {time.value:.0f} d)"
    )

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.02)
cb.set_label("Time (days)")
ax.set(
    xlabel="Frequency (GHz)",
    ylabel=r"$F_\nu$ (mJy)",
    title=r"SSA + FFA time evolution ($\dot{M}/v_w = 10^{-9}\ M_\odot\ {\rm yr}^{-1}\ ({\rm km/s})^{-1}$)",
    xlim=(1e-1, 1e4),
    ylim=(1e-1, 1e3),
)
ax.grid(True, which="both", ls="--", alpha=0.3)

plt.tight_layout()
plt.show()